In [ ]:
from datetime import datetime
import pyspark.sql.functions as F

for chave, valor in {
    "data_inicio": "20260501",
    "data_fim": "20260601",
    "catalogo": "workspace",
    "esquema": "default"
}.items():
    dbutils.widgets.text(chave, valor)

inicio = dbutils.widgets.get("data_inicio")
fim = dbutils.widgets.get("data_fim")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")

datetime.strptime(inicio, "%Y%m%d")
datetime.strptime(fim, "%Y%m%d")

if inicio >= fim:
    raise ValueError("Periodo invalido.")

tabela = f"{catalogo}.{esquema}.log_monitoramento_drift_aprimorado"

logs = spark.table(tabela)

total_drift = logs.filter(
    (F.col("periodo_inicio") == inicio) &
    (F.col("periodo_fim") == fim) &
    (F.col("drift_detectado") == True)
).count()

dbutils.jobs.taskValues.set(
    key="total_drift",
    value=int(total_drift)
)

print(f"Periodo: {inicio} a {fim} (fim exclusivo)")
print(f"Drifts encontrados: {total_drift}")

if total_drift > 0:
    print("Condicao verdadeira: re-treino autorizado.")
else:
    print("Condicao falsa: re-treino nao necessario.")